# 03 · Shares por clase, D-CPI y comparación de salarios reales

Última pieza del POC. Tres pasos:
1. Para cada hogar de la ENGH con al menos un ocupado, asignar la clase del **ocupado de mayor ingreso** (proxy del 'jefe económico'). Calcular shares promedio por clase.
2. Cargar (o sintetizar, en este POC) un IPC trimestral por división. Construir el **D-CPI por clase** = Σ shares × IPC división.
3. Deflactar los salarios EPH por clase con (a) el IPC oficial y (b) el D-CPI propio. Comparar.

**Por qué el ocupado de mayor ingreso y no el jefe del hogar**: el jefe en ENGH puede ser inactivo (jubilado, ama de casa). Para que el matching tenga sentido, asignamos la clase de quien efectivamente está ocupado y aporta más al hogar. Es una decisión simplificadora documentable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

hog = pd.read_parquet("data/engh_hogares.parquet")
ocu = pd.read_parquet("data/engh_ocupados_with_lca.parquet")

CLASS_NAMES = {
    0: "Obreros formales (sec.)",
    1: "Asalariados informales jóvenes",
    2: "Trabajadores mayores con primario",
    3: "Profesionales asalariadas",
    4: "Cuentapropistas y patrones",
}

## Shares por clase

In [ ]:
ocu_sorted = ocu.sort_values(["id","ingtotp"], ascending=[True, False])
hog_class = ocu_sorted.groupby("id").first()[["LCA_class_pred"]].reset_index()
hog_class = hog_class.rename(columns={"LCA_class_pred":"LCA_class"})
hog_lab = hog.merge(hog_class, on="id", how="inner")
print(f"Hogares con al menos un ocupado: {len(hog_lab)} de {len(hog)}")

share_cols = [f"share_{i:02d}" for i in range(1,13)]
def wmean(g, cols, w):
    W = g[w].sum()
    return pd.Series({c: (g[c]*g[w]).sum()/W for c in cols})

shares_by_class = hog_lab.groupby("LCA_class").apply(
    lambda g: wmean(g, share_cols, "pondera"))
shares_by_class.to_csv("data/shares_by_class.csv")
print("\nShares promedio por clase (cada fila suma 1):")
print(shares_by_class.round(3))

### Visualización

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_df = shares_by_class.copy()
plot_df.index = [CLASS_NAMES[i] for i in plot_df.index]
plot_df.columns = ["Aliment.","Beb.alc/Tabaco","Indument.","Vivienda","Equip.hogar",
                   "Salud","Transporte","Comunic.","Recreación","Educación",
                   "Restaur.","Varios"]
plot_df.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_title("Estructura de gasto por clase latente (ENGH 2017/18)")
ax.set_ylabel("Share del gasto total"); ax.set_xlabel("")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.xticks(rotation=20, ha="right")
fig.tight_layout()
fig.savefig("figures/02_shares_by_class.png", dpi=120)
plt.show()

## IPC por división — INDEC real

Cargamos la serie mensual del INDEC con apertura por las 12 divisiones COICOP, base diciembre 2016 = 100. Pasos:

1. **Renombrar columnas** del CSV INDEC al formato `div_01..div_12` que usa el resto del POC. El orden del CSV coincide 1:1 con el orden COICOP de la ENGH.
2. **Filtrar a 2017-01-01 en adelante**, agregar a trimestres por promedio.
3. **Rebasear a 2017Q1 = 100** para que la base coincida con el resto del POC.
4. **Sanity check**: reconstruir el IPC oficial usando los `share` promedio de la ENGH y compararlo con la columna `nivel_general`. Si la diferencia es chica (<1%), valida que las shares de la ENGH están bien calculadas y que las divisiones se mapean correctamente.

In [ ]:
ipc_raw = pd.read_csv("data/ipc_indec_real.csv", parse_dates=["indice_tiempo"])

col_map = {
    "ipc_2016_alimentos_bebidas": "div_01",
    "ipc_2016_bebidas_alcoholicas": "div_02",
    "ipc_2016_indumentaria": "div_03",
    "ipc_2016_vivienda": "div_04",
    "ipc_2016_equipamiento_del_hogar": "div_05",
    "ipc_2016_atencion_medica_salud": "div_06",
    "ipc_2016_transporte": "div_07",
    "ipc_2016_informacion_comunicacion": "div_08",
    "ipc_2016_recreacion_cultura": "div_09",
    "ipc_2016_educacion": "div_10",
    "ipc_2016_restauranes_hoteles": "div_11",
    "ipc_2016_bienes_servicios_varios": "div_12",
}
ipc_m = ipc_raw[["indice_tiempo"] + list(col_map.keys())].rename(columns=col_map)
ipc_m["nivel_general"] = ipc_raw["ipc_2016_nivel_general"]
ipc_m = ipc_m[ipc_m["indice_tiempo"] >= "2017-01-01"].copy()
ipc_m["trimestre"] = ipc_m["indice_tiempo"].dt.to_period("Q").astype(str)

ipc = (ipc_m.groupby("trimestre")[list(col_map.values()) + ["nivel_general"]]
            .mean().sort_index())

# Rebasear a 2017Q1 = 100
base = ipc.iloc[0].copy()
for c in ipc.columns:
    ipc[c] = ipc[c] / base[c] * 100
ipc.to_csv("data/ipc_real_quarterly.csv")
print(f"IPC INDEC trimestral, base 2017Q1=100. {len(ipc)} trimestres ({ipc.index[0]} → {ipc.index[-1]})")
print(ipc.iloc[[0, 8, 16, 24, -1]].round(0))

# Sanity check: el IPC oficial reconstruido a partir de las shares debería pegar bien
overall_shares = np.array([
    (hog_lab[f"share_{i:02d}"]*hog_lab["pondera"]).sum() / hog_lab["pondera"].sum()
    for i in range(1,13)
])
ipc_reconstr = ipc[[f"div_{i:02d}" for i in range(1,13)]].values @ overall_shares
diff_pct = (ipc_reconstr[-1] / ipc["nivel_general"].iloc[-1] - 1) * 100
print(f"\nSanity check: IPC oficial reconstruido vs publicado, último trim: {diff_pct:+.2f}%")
print("Si esta diferencia es chica (<1-2%), las shares y el mapeo de divisiones están OK.")

## D-CPI por clase

Para cada clase, el D-CPI es Σ_d (share_d × IPC_d). Como sanity, también calculamos el "IPC oficial reconstruido" (con shares promedio del país) y lo comparamos con el `nivel_general` publicado por el INDEC.

In [ ]:
ipc_cols = [f"div_{i:02d}" for i in range(1,13)]
M = ipc[ipc_cols].values   # n_t x 12

dcpi = pd.DataFrame(index=ipc.index)
for c in shares_by_class.index:
    s = np.array([shares_by_class.loc[c, f"share_{i:02d}"] for i in range(1,13)])
    dcpi[f"class_{c}"] = M @ s
dcpi["IPC_oficial"] = ipc["nivel_general"].values  # publicado, no reconstruido
dcpi.to_csv("data/dcpi_by_class.csv")
print("Inflación acumulada 2017Q1 → último trimestre disponible:")
print(((dcpi.iloc[-1] / 100 - 1) * 100).round(1))

## Salarios reales por clase — IPC oficial vs D-CPI propio

In [ ]:
eph = pd.read_parquet("data/eph_with_lca.parquet")
eph["trimestre"] = eph["ANO4"].astype(int).astype(str) + "Q" + eph["TRIMESTRE"].astype(int).astype(str)
def wmean_s(g, col, w):
    W = g[w].sum()
    return (g[col]*g[w]).sum() / W
wages = eph.groupby(["trimestre","LCA_class"]).apply(
    lambda g: wmean_s(g, "P21", "PONDERA")).unstack()
wages.columns = [f"class_{c}" for c in wages.columns]
wages = wages.sort_index()

common = wages.index.intersection(dcpi.index)
wages = wages.loc[common]; dcpi_a = dcpi.loc[common]

real_own = pd.DataFrame(index=common)
real_off = pd.DataFrame(index=common)
for c in range(5):
    real_own[f"class_{c}"] = wages[f"class_{c}"] / dcpi_a[f"class_{c}"] * 100
    real_off[f"class_{c}"] = wages[f"class_{c}"] / dcpi_a["IPC_oficial"] * 100

real_own_idx = real_own.div(real_own.iloc[0]) * 100
real_off_idx = real_off.div(real_off.iloc[0]) * 100
real_own_idx.to_csv("data/real_wages_own_dcpi.csv")
real_off_idx.to_csv("data/real_wages_official_ipc.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for c in range(5):
    axes[0].plot(real_off_idx.index, real_off_idx[f"class_{c}"], lw=1.8, label=CLASS_NAMES[c])
    axes[1].plot(real_own_idx.index, real_own_idx[f"class_{c}"], lw=1.8, label=CLASS_NAMES[c])
axes[0].set_title("Deflactado por IPC OFICIAL (canasta promedio)")
axes[1].set_title("Deflactado por D-CPI de cada clase")
for ax in axes:
    ax.axhline(100, ls="--", color="black", alpha=0.3)
    ax.set_ylabel("Salario real (índice 2017Q1=100)")
    ax.grid(alpha=0.3)
    step = max(1, len(real_off_idx)//9)
    ax.set_xticks(real_off_idx.index[::step])
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
axes[1].legend(loc="upper right", fontsize=8)
fig.suptitle("Salarios reales por perfil — IPC oficial vs D-CPI propio",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig("figures/03_real_wages_comparison.png", dpi=120)
plt.show()

## Money chart: ¿quién perdió cuánto?

In [ ]:
loss_off = (real_off_idx.iloc[-1] - 100)
loss_own = (real_own_idx.iloc[-1] - 100)

fig, ax = plt.subplots(figsize=(11, 5))
xpos = np.arange(5); w = 0.4
b1 = ax.bar(xpos - w/2, loss_off.values, w, label="Deflactando con IPC oficial", color="#888")
b2 = ax.bar(xpos + w/2, loss_own.values, w, label="Deflactando con D-CPI propio", color="#4080c0")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(xpos)
ax.set_xticklabels([CLASS_NAMES[i] for i in range(5)], rotation=15, ha="right")
ax.set_ylabel("% cambio salario real 2017Q1 → 2025Q3")
ax.set_title("¿Cuánto perdió cada perfil? — IPC INDEC real")
ax.legend(); ax.grid(alpha=0.3, axis="y")
for bars in [b1, b2]:
    for b in bars:
        h = b.get_height()
        ax.annotate(f"{h:.0f}%", xy=(b.get_x()+b.get_width()/2, h),
                    xytext=(0, 3 if h>=0 else -12), textcoords="offset points",
                    ha="center", fontsize=8)
fig.tight_layout()
fig.savefig("figures/04_money_chart.png", dpi=120)
plt.show()

print("\nResumen final — pérdida salario real 2017Q1 → 2025Q3:")
result = pd.DataFrame({"IPC_oficial": loss_off, "D-CPI_propio": loss_own}).round(1)
result.index = [CLASS_NAMES[i] for i in range(5)]
print(result)

## Limitaciones (declaración honesta)

1. **CIA.** El matching asume independencia condicional y no se testea directamente.
2. **Asimetría rica/pobre.** El LCA usa 6 variables; el matching, 5. La variable `FORMAL` (formalidad) no entra al matching por falta de equivalente directo en ENGH.
3. **Asignación al hogar vía ocupado de mayor ingreso.** Es una simplificación; no siempre coincide con el jefe declarado.
4. **Canasta estática.** La estructura de gasto se asume constante 2017–2025 (la ENGH es una sola foto).
5. **Pesos replicados sin uso.** No se usan los 200 pesos replicados de la ENGH para errores estándar formales. Los D-CPI son puntuales.
6. **Sub-reporte de ingresos.** Cuentapropistas y patrones tienden a sub-declarar ingresos en EPH; los resultados de la clase 4 hay que tomarlos con cautela.